# Tugas 1 — Milestone 3: Analisis Temporal dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Abdul Gofir Samjani  
**NIM:** 202310370311136  
**Username GitHub:** abdulgofirsamjani

---

Notebook ini melanjutkan hasil cleaning Milestone 2 pada `data/processed/indonesian_news_az_clean.parquet`. Seluruh statistik dan insight dihitung ulang dari file lokal ketika notebook dijalankan. Baris dengan tanggal yang tidak berhasil diparsing tetap ada di dataset utama, tetapi dikecualikan dari perhitungan berbasis waktu karena tidak memiliki tanggal kalender yang dapat dipertanggungjawabkan. Dataset ini tidak memuat koordinat geografis; analisis multidimensi karena itu menggunakan panjang judul, panjang artikel, dan domain sumber, bukan peta spasial.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

File hasil cleaning dibaca dengan Polars lazy scan. Path dicari dari direktori kerja saat ini hingga direktori induknya agar notebook dapat dijalankan dari root repository maupun dari `notebooks/`. Pemeriksaan ringkas memvalidasi skema, jumlah baris, nilai tanggal, dan null tanpa memuat keseluruhan dataset ke Pandas.

In [3]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents)
     if (candidate / "data" / "processed" / "indonesian_news_az_clean.parquet").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Tidak menemukan data/processed/indonesian_news_az_clean.parquet "
        "dari direktori kerja saat ini atau induknya."
    )

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "indonesian_news_az_clean.parquet"
news = pl.scan_parquet(str(DATA_PATH), low_memory=True)
schema = news.collect_schema()
required_columns = {
    "date", "link", "title", "text", "date_parsed", "date_status",
    "title_length", "text_length",
}
missing_columns = required_columns.difference(schema.names())
if missing_columns:
    raise ValueError(f"Kolom hasil cleaning tidak lengkap: {sorted(missing_columns)}")
if schema["date_parsed"] != pl.Date:
    raise TypeError(f"date_parsed harus bertipe Date, ditemukan {schema['date_parsed']}")

summary = news.select([
    pl.len().alias("rows"),
    pl.col("date_parsed").is_not_null().sum().alias("valid_dates"),
    pl.col("date_parsed").is_null().sum().alias("invalid_dates"),
    pl.col("link").is_null().sum().alias("null_link"),
    pl.col("title").is_null().sum().alias("null_title"),
    pl.col("text").is_null().sum().alias("null_text"),
    pl.col("date_parsed").min().alias("first_valid_date"),
    pl.col("date_parsed").max().alias("last_valid_date"),
]).collect(engine="streaming").row(0, named=True)
row_count = summary["rows"]
valid_count = summary["valid_dates"]
invalid_count = summary["invalid_dates"]
file_size_bytes = DATA_PATH.stat().st_size

print(f"Root repository: {PROJECT_ROOT}")
print(f"Sumber data: {DATA_PATH.relative_to(PROJECT_ROOT)} ({file_size_bytes:,} byte)")
print(f"Ukuran data: {row_count:,} baris × {len(schema)} kolom")
print("Skema:", dict(schema))
print(f"Tanggal valid: {valid_count:,}; date_parsed null/tidak valid: {invalid_count:,}")
print(f"Rentang tanggal valid: {summary['first_valid_date']} sampai {summary['last_valid_date']}")
print(f"Null link/title/text: {summary['null_link']:,}/{summary['null_title']:,}/{summary['null_text']:,}")

# Alias eksplisit untuk alur analisis temporal; invalid rows tetap tersedia dalam `news`.
news_valid_dates = news.filter(pl.col("date_parsed").is_not_null())
news_invalid_dates = news.filter(pl.col("date_parsed").is_null())
print(f"Baris dipakai analisis temporal: {news_valid_dates.select(pl.len()).collect(engine='streaming').item():,}")
print(f"Baris tanpa tanggal kalender untuk grafik temporal: {news_invalid_dates.select(pl.len()).collect(engine='streaming').item():,}")

Root repository: <root repository>
Sumber data: data\processed\indonesian_news_az_clean.parquet (803,849,540 byte)
Ukuran data: 1,029,485 baris × 8 kolom
Skema: {'date': String, 'link': String, 'title': String, 'text': String, 'date_parsed': Date, 'date_status': String, 'title_length': UInt32, 'text_length': UInt32}
Tanggal valid: 928,082; date_parsed null/tidak valid: 101,403
Rentang tanggal valid: 2019-05-10 sampai 2023-10-01
Null link/title/text: 0/0/0
Baris dipakai analisis temporal: 928,082
Baris tanpa tanggal kalender untuk grafik temporal: 101,403


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Tanggal valid dikelompokkan ke bulan kalender yang benar-benar memiliki artikel dengan `group_by_dynamic()`. Moving average tiga bulan dihitung setelah pengurutan kronologis; `shift(1)` membandingkan volume dengan bulan observasi sebelumnya. Tidak dibuat baris tanggal atau bulan sintetis. Karena itu, jeda antar-bulan yang teramati tidak dapat langsung ditafsirkan sebagai bulan tanpa artikel: bisa jadi cakupan sumber tidak tersedia. Rata-rata bergerak dan perbandingan ini menggambarkan dataset, bukan seluruh media Indonesia.

DuckDB menghitung moving average tiga observasi sebagai pemeriksaan silang terhadap hasil Polars.

In [4]:
monthly = (
    news_valid_dates
    .select("date_parsed")
    .sort("date_parsed")
    .group_by_dynamic("date_parsed", every="1mo", closed="left", label="left")
    .agg(pl.len().cast(pl.Int64).alias("article_count"))
    .sort("date_parsed")
    .with_columns([
        pl.col("article_count").rolling_mean(window_size=3, min_samples=1).alias("moving_avg_3_observed_months"),
        pl.col("article_count").shift(1).alias("previous_observed_month_count"),
    ])
    .with_columns(
        (pl.col("article_count") - pl.col("previous_observed_month_count")).alias("change_from_previous_observed_month")
    )
    .collect(engine="streaming")
)
if monthly.is_empty():
    raise ValueError("Tidak ada tanggal valid untuk agregasi temporal.")

# Pastikan delta bertanda menangani penurunan dan kenaikan tanpa integer underflow.
delta_check = pl.DataFrame({"count": [5, 2, 9]}).with_columns(
    pl.col("count").cast(pl.Int64).shift(1).alias("previous")
).with_columns((pl.col("count") - pl.col("previous")).alias("delta"))
assert delta_check.get_column("delta").to_list() == [None, -3, 7]

# Pembanding SQL: 3 baris observasi sebelumnya/terkini, bukan kalender sintetis.
duck = duckdb.connect(database=":memory:")
duck.register("monthly_counts", monthly.select("date_parsed", "article_count").to_arrow())
duckdb_cursor = duck.execute("""
    SELECT date_parsed, article_count,
           AVG(article_count) OVER (
               ORDER BY date_parsed ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
           ) AS moving_avg_duckdb
    FROM monthly_counts
    ORDER BY date_parsed
""")
duckdb_moving = pl.DataFrame(
    duckdb_cursor.fetchall(),
    schema=["date_parsed", "article_count", "moving_avg_duckdb"],
    orient="row",
)
duck.close()
max_window_difference = max(
    abs(a - b)
    for a, b in zip(monthly.get_column("moving_avg_3_observed_months"), duckdb_moving.get_column("moving_avg_duckdb"))
)
assert max_window_difference < 1e-8, f"Hasil moving average Polars/DuckDB berbeda: {max_window_difference}"

first_month = monthly.item(0, "date_parsed")
last_month = monthly.item(monthly.height - 1, "date_parsed")
calendar_month_span = (last_month.year - first_month.year) * 12 + last_month.month - first_month.month + 1
observed_month_count = monthly.height
peak_month = monthly.sort("article_count", descending=True).row(0, named=True)
median_monthly_count = monthly.get_column("article_count").median()
mean_monthly_count = monthly.get_column("article_count").mean()
print(f"Bulan observasi: {observed_month_count} dari rentang kalender {calendar_month_span} bulan.")
print(f"Observasi pertama/terakhir: {first_month} sampai {last_month}.")
print(f"Bulan dengan artikel terbanyak: {peak_month['date_parsed']} ({peak_month['article_count']:,} artikel).")
print(f"Median/rata-rata artikel per bulan observasi: {median_monthly_count:,.0f}/{mean_monthly_count:,.1f}.")
print(f"Moving average Polars vs DuckDB: selisih absolut maksimum {max_window_difference:.3g}.")
monthly.tail(5)

Bulan observasi: 22 dari rentang kalender 54 bulan.
Observasi pertama/terakhir: 2019-05-01 sampai 2023-10-01.
Bulan dengan artikel terbanyak: 2023-08-01 (117,163 artikel).
Median/rata-rata artikel per bulan observasi: 31,612/42,185.5.
Moving average Polars vs DuckDB: selisih absolut maksimum 0.


date_parsed,article_count,moving_avg_3_observed_months,previous_observed_month_count,change_from_previous_observed_month
date,u32,f64,u32,u32
2023-06-01,101795,87024.0,90033,11762
2023-07-01,108801,100209.666667,101795,7006
2023-08-01,117163,109253.0,108801,8362
2023-09-01,105019,110327.666667,117163,4294955152
2023-10-01,1114,74432.0,105019,4294863391


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly)

Semua agregasi dilakukan dengan Polars sebelum data ringkas dipindahkan ke Plotly. Grafik titik memakai sampel deterministik setiap 200 baris pemindaian (sekitar 0,5% baris), sehingga hasilnya dapat diulang dan tetap responsif. Plot tidak memuat teks isi artikel.

### Visualisasi 1 — Tren Publikasi Artikel

Jumlah artikel per bulan observasi dan moving average tiga observasi. Range slider membantu meninjau periode. Bulan yang tidak muncul tidak diisi nol karena tidak dapat dibedakan dari periode tanpa cakupan data.

In [5]:
monthly_pd = monthly.to_pandas()
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=monthly_pd["date_parsed"], y=monthly_pd["article_count"],
    mode="lines+markers", name="Artikel per bulan observasi",
    hovertemplate="Bulan: %{x|%Y-%m}<br>Artikel: %{y:,}<extra></extra>",
))
fig1.add_trace(go.Scatter(
    x=monthly_pd["date_parsed"], y=monthly_pd["moving_avg_3_observed_months"],
    mode="lines", name="Moving average (3 observasi)",
    line={"width": 3},
    hovertemplate="Bulan: %{x|%Y-%m}<br>MA: %{y:,.1f}<extra></extra>",
))
fig1.update_layout(title="Tren Publikasi Artikel per Bulan Observasi", xaxis_title="Bulan", yaxis_title="Jumlah artikel", hovermode="x unified", legend_title="Seri")
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

### Visualisasi 2 — Distribusi Panjang Artikel

Histogram menggunakan sampel deterministik agar lebih ringan. Sumbu dibatasi pada persentil ke-99 untuk memperjelas massa utama distribusi; statistik median dan ambang persentil dihitung dari seluruh dataset. Ekor di atas batas tetap dihitung, bukan dibuang.

In [6]:
length_summary = news.select([
    pl.col("title_length").median().alias("title_median"),
    pl.col("text_length").median().alias("text_median"),
    pl.col("text_length").quantile(0.99, interpolation="nearest").alias("text_p99"),
    pl.col("text_length").quantile(0.999, interpolation="nearest").alias("text_p999"),
    (pl.col("text_length") > pl.col("text_length").quantile(0.99, interpolation="nearest")).sum().alias("text_above_p99"),
]).collect(engine="streaming").row(0, named=True)
length_sample = (
    news.with_row_index("scan_row")
    .filter(pl.col("scan_row") % 200 == 0)
    .select("text_length")
    .collect(engine="streaming")
)
fig2 = px.histogram(length_sample.to_pandas(), x="text_length", nbins=60,
                    title="Distribusi Panjang Artikel (Sampel Deterministik)",
                    labels={"text_length": "Panjang artikel (karakter)", "count": "Jumlah pada sampel"},
                    color_discrete_sequence=["#2563eb"])
fig2.update_xaxes(range=[0, length_summary["text_p99"]])
fig2.update_layout(yaxis_title="Jumlah artikel pada sampel", bargap=0.05)
fig2.show()

### Visualisasi 3 — Korelasi Fitur Numerik

Korelasi Pearson dihitung dari seluruh pasangan `title_length` dan `text_length`. Keduanya memiliki interpretasi langsung sebagai panjang karakter. Korelasi tidak membuktikan kausalitas dan tidak mengukur kualitas atau keterbacaan artikel.

In [7]:
correlation = news.select(pl.corr("title_length", "text_length").alias("pearson_r")).collect(engine="streaming").item()
corr_matrix = [[1.0, correlation], [correlation, 1.0]]
fig3 = go.Figure(data=go.Heatmap(
    z=corr_matrix, x=["Panjang judul", "Panjang artikel"], y=["Panjang judul", "Panjang artikel"],
    zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
    text=[[f"{v:.3f}" for v in row] for row in corr_matrix], texttemplate="%{text}",
    hovertemplate="%{y} × %{x}<br>Korelasi Pearson: %{z:.3f}<extra></extra>", colorbar_title="r",
))
fig3.update_layout(title="Korelasi Panjang Judul dan Panjang Artikel", xaxis_title="Fitur", yaxis_title="Fitur")
fig3.show()

### Visualisasi 4 — Komposisi Sumber Berita

Hostname diambil dari URL HTTP(S), diubah menjadi huruf kecil, dan awalan `www.` dihilangkan. Bar menampilkan 15 domain dengan artikel terbanyak. Hostname adalah petunjuk sumber pada URL, bukan verifikasi afiliasi penerbit.

In [8]:
domain_expr = (
    pl.col("link").str.extract(r"(?i)^https?://(?:www\.)?([^/:?#]+)", 1)
    .str.to_lowercase().alias("domain")
)
domain_counts = (
    news.select(domain_expr).filter(pl.col("domain").is_not_null())
    .group_by("domain").agg(pl.len().alias("article_count"))
    .sort("article_count", descending=True).collect(engine="streaming")
)
domain_total = domain_counts.get_column("article_count").sum()
top_domains = domain_counts.head(15).with_columns(
    (pl.col("article_count") / domain_total * 100).alias("share_percent")
)
fig4 = px.bar(top_domains.to_pandas(), x="article_count", y="domain", orientation="h",
              color="share_percent", color_continuous_scale="Blues",
              title="15 Domain dengan Artikel Terbanyak",
              labels={"article_count": "Jumlah artikel", "domain": "Hostname", "share_percent": "Porsi dari URL terdeteksi (%)"},
              hover_data={"share_percent": ":.2f"})
fig4.update_layout(yaxis={"categoryorder": "total ascending"})
fig4.show()

### Visualisasi 5 — Pola Temporal Berkala

Heatmap menghitung artikel menurut bulan kalender (1–12) dan hari dalam minggu dari `date_parsed`. Sel bernilai nol berarti kombinasi tersebut tidak memiliki baris valid dalam data yang tersedia; cakupan sumber yang tidak merata dapat memengaruhi pola.

In [9]:
weekday_month = (
    news_valid_dates.select([
        pl.col("date_parsed").dt.month().alias("month"),
        pl.col("date_parsed").dt.weekday().alias("weekday_iso"),
    ])
    .group_by(["month", "weekday_iso"]).agg(pl.len().alias("article_count"))
    .collect(engine="streaming")
)
weekday_names = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
heatmap_values = [[0 for _ in range(12)] for _ in range(7)]
for row in weekday_month.iter_rows(named=True):
    heatmap_values[row["weekday_iso"] - 1][row["month"] - 1] = row["article_count"]
fig5 = go.Figure(data=go.Heatmap(
    z=heatmap_values, x=["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"],
    y=weekday_names, colorscale="Viridis", colorbar_title="Artikel",
    hovertemplate="%{y}, %{x}<br>Artikel bertanggal valid: %{z:,}<extra></extra>",
))
fig5.update_layout(title="Distribusi Artikel menurut Hari dalam Minggu dan Bulan", xaxis_title="Bulan kalender", yaxis_title="Hari (ISO)")
fig5.show()

### Visualisasi 6 — Analisis Multidimensi

Scatter plot membandingkan panjang judul dan isi pada sampel deterministik sekitar 0,5% baris (setiap baris ke-200 dari urutan scan). Warna menunjukkan domain teratas; domain lain dan URL yang tidak dikenali dikelompokkan sebagai lainnya/tidak diketahui. Sampel cocok untuk melihat sebaran, tetapi tidak untuk menghitung proporsi populasi.

In [10]:
top_domain_names = top_domains.get_column("domain").to_list()
scatter_sample = (
    news.with_row_index("scan_row")
    .with_columns(domain_expr)
    .filter(pl.col("scan_row") % 200 == 0)
    .select("title_length", "text_length", "domain")
    .with_columns(
        pl.when(pl.col("domain").is_in(top_domain_names)).then(pl.col("domain"))
        .when(pl.col("domain").is_null()).then(pl.lit("Tidak diketahui"))
        .otherwise(pl.lit("Lainnya")).alias("domain_group")
    )
    .collect(engine="streaming")
)
fig6 = px.scatter(scatter_sample.to_pandas(), x="title_length", y="text_length", color="domain_group",
                  opacity=0.55, render_mode="webgl", title="Panjang Judul vs Panjang Artikel (Sampel 0,5%)",
                  labels={"title_length": "Panjang judul (karakter)", "text_length": "Panjang artikel (karakter)", "domain_group": "Kelompok domain"},
                  hover_data={"domain_group": True})
fig6.update_layout(legend_title="Domain sumber")
fig6.show()
print(f"Ukuran sampel scatter: {scatter_sample.height:,} baris dari {row_count:,} ({scatter_sample.height / row_count:.2%}).")

Ukuran sampel scatter: 5,148 baris dari 1,029,485 (0.50%).


In [ ]:
from pathlib import Path
import re

FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figures = {
    "01_tren_publikasi.html": fig1,
    "02_distribusi_panjang_artikel.html": fig2,
    "03_korelasi_fitur.html": fig3,
    "04_top_15_domain.html": fig4,
    "05_heatmap_temporal.html": fig5,
    "06_scatter_multidimensi.html": fig6,
}
for filename, figure in figures.items():
    target = FIGURE_DIR / filename
    figure.write_html(target, include_plotlyjs="directory", full_html=True, auto_open=False)
    html = target.read_text(encoding="utf-8")
    assert 'class="plotly-graph-div"' in html, f"Grafik Plotly tidak ditemukan: {target}"
    assert re.search(r'<script[^>]+src=[\"\']plotly\.min\.js', html), f"Referensi Plotly lokal tidak ada: {target}"
    assert not re.search(r'https?://[^\" ]*(?:plotly|cdn)', html, re.I), f"Grafik bergantung CDN: {target}"
    assert (FIGURE_DIR / "plotly.min.js").is_file(), "plotly.min.js lokal tidak ditemukan"
print(f"Enam grafik interaktif diekspor dan divalidasi di {FIGURE_DIR}")


## Bagian 4: Rumusan 5 Insight Analitik Utama

Lima temuan di bawah dibentuk dari agregasi yang dihitung pada sesi eksekusi ini. Jalankan ulang notebook untuk memperbarui angka jika file hasil cleaning berubah. Rekomendasi dibatasi oleh cakupan dataset; tidak ada klaim kausalitas.

In [11]:
# Ringkasan segmentasi domain: porsi host dihitung hanya pada URL yang dapat dikenali.
top_domain_row = domain_counts.row(0, named=True)
top_domain_share = top_domain_row["article_count"] / domain_total
second_domain_row = domain_counts.row(1, named=True) if domain_counts.height > 1 else None

weekday_totals = weekday_month.group_by("weekday_iso").agg(pl.col("article_count").sum()).sort("weekday_iso")
weekday_peak = weekday_totals.sort("article_count", descending=True).row(0, named=True)
invalid_share = invalid_count / row_count
above_p99_count = length_summary["text_above_p99"]

insights = [
    (
        "1. Tren Waktu dan Musiman",
        f"Bulan observasi dengan artikel terbanyak adalah **{peak_month['date_parsed']}** ({peak_month['article_count']:,} artikel); median antarbulan observasi {median_monthly_count:,.0f}. Moving average tiga observasi dihitung dan hasilnya cocok dengan DuckDB (selisih maksimum {max_window_difference:.3g}).",
        "Bukti: Visualisasi 1 dan agregasi `group_by_dynamic()`; tersedia "
        f"{observed_month_count} bulan observasi dari rentang kalender {calendar_month_span} bulan. "
        "Bulan yang tidak teramati tidak diisi nol, sehingga jeda tidak membuktikan tidak ada publikasi. Ini bukan tren seluruh media Indonesia."
    ),
    (
        "2. Distribusi dan Anomali",
        f"Median panjang artikel adalah {length_summary['text_median']:,.0f} karakter; persentil ke-99 {length_summary['text_p99']:,.0f} karakter. Sebanyak {above_p99_count:,} baris berada di atas ambang P99 yang dihitung dari seluruh data.",
        "Bukti: Visualisasi 2 dan kuantil seluruh data. Nilai di atas P99 disebut ekor panjang/kandidat ekstrem, bukan kesalahan; histogram hanya sampel dan dibatasi sampai P99 untuk keterbacaan."
    ),
    (
        "3. Korelasi Kunci",
        f"Korelasi Pearson panjang judul dan panjang artikel adalah **r = {correlation:.3f}** pada seluruh {row_count:,} baris.",
        "Bukti: Visualisasi 3. Korelasi mengukur asosiasi linear, tidak menyatakan sebab-akibat dan dapat melewatkan hubungan non-linear; panjang bukan ukuran mutu berita."
    ),
    (
        "4. Perilaku Kategori atau Segmen",
        f"Hostname terbanyak adalah **{top_domain_row['domain']}** dengan {top_domain_row['article_count']:,} artikel ({top_domain_share:.2%} dari {domain_total:,} URL ber-hostname yang terdeteksi). Hari ISO dengan hitungan tertinggi pada tanggal valid adalah {weekday_names[weekday_peak['weekday_iso'] - 1]} ({weekday_peak['article_count']:,} artikel).",
        "Bukti: Visualisasi 4 dan 5. Hostname tidak selalu identik dengan identitas penerbit; hitungan hari dipengaruhi volume cakupan tiap sumber dan tanggal invalid dikecualikan."
    ),
    (
        "5. Rekomendasi Praktis",
        f"Pertahankan penanda tanggal invalid dan prioritaskan audit sumber/tata format tanggal: {invalid_count:,} dari {row_count:,} baris ({invalid_share:.2%}) tidak memiliki `date_parsed` yang valid. Untuk pemantauan sumber, laporkan porsi domain teratas ({top_domain_share:.2%}) secara berkala bersama jumlah URL yang dapat dikenali.",
        "Bukti: ringkasan kualitas data dan Visualisasi 4. Jangan mengimputasi tanggal tebakan; dokumentasikan cakupan sumber dan validasi tren sebelum menggeneralisasi hasil ke ekosistem media Indonesia."
    ),
]
for heading, finding, caveat in insights:
    display(Markdown(f"### {heading}\n\n{finding}\n\n{caveat}"))

### 1. Tren Waktu dan Musiman

Bulan observasi dengan artikel terbanyak adalah **2023-08-01** (117,163 artikel); median antarbulan observasi 31,612. Moving average tiga observasi dihitung dan hasilnya cocok dengan DuckDB (selisih maksimum 0).

Bukti: Visualisasi 1 dan agregasi `group_by_dynamic()`; tersedia 22 bulan observasi dari rentang kalender 54 bulan. Bulan yang tidak teramati tidak diisi nol, sehingga jeda tidak membuktikan tidak ada publikasi. Ini bukan tren seluruh media Indonesia.

### 2. Distribusi dan Anomali

Median panjang artikel adalah 2,134 karakter; persentil ke-99 7,145 karakter. Sebanyak 10,293 baris berada di atas ambang P99 yang dihitung dari seluruh data.

Bukti: Visualisasi 2 dan kuantil seluruh data. Nilai di atas P99 disebut ekor panjang/kandidat ekstrem, bukan kesalahan; histogram hanya sampel dan dibatasi sampai P99 untuk keterbacaan.

### 3. Korelasi Kunci

Korelasi Pearson panjang judul dan panjang artikel adalah **r = 0.126** pada seluruh 1,029,485 baris.

Bukti: Visualisasi 3. Korelasi mengukur asosiasi linear, tidak menyatakan sebab-akibat dan dapat melewatkan hubungan non-linear; panjang bukan ukuran mutu berita.

### 4. Perilaku Kategori atau Segmen

Hostname terbanyak adalah **tribunnews.com** dengan 107,817 artikel (10.47% dari 1,029,485 URL ber-hostname yang terdeteksi). Hari ISO dengan hitungan tertinggi pada tanggal valid adalah Selasa (155,745 artikel).

Bukti: Visualisasi 4 dan 5. Hostname tidak selalu identik dengan identitas penerbit; hitungan hari dipengaruhi volume cakupan tiap sumber dan tanggal invalid dikecualikan.

### 5. Rekomendasi Praktis

Pertahankan penanda tanggal invalid dan prioritaskan audit sumber/tata format tanggal: 101,403 dari 1,029,485 baris (9.85%) tidak memiliki `date_parsed` yang valid. Untuk pemantauan sumber, laporkan porsi domain teratas (10.47%) secara berkala bersama jumlah URL yang dapat dikenali.

Bukti: ringkasan kualitas data dan Visualisasi 4. Jangan mengimputasi tanggal tebakan; dokumentasikan cakupan sumber dan validasi tren sebelum menggeneralisasi hasil ke ekosistem media Indonesia.